In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from src.strategy import (
    calculate_z_score,
    determine_entry_signal,
    calculate_stop_loss,
    calculate_liquidation_boundary,
    calculate_max_dte,
    generate_signal,
    signal_to_label
)

In [12]:
ou_results = pd.read_parquet("data/processed/selected_pairs.parquet")

ou_results.head()

,pair,dependent,independent,mean,std,min spread,max spread,theta,half_life,expected_convergence,...,stock1_has_options,stock2_has_options,stock1_num_expirations,stock2_num_expirations,stock1_first_expiration,stock2_first_expiration,stock1_last_expiration,stock2_last_expiration,eligible,exclusion_reason
0,V-MA,V,MA,-4.110665e-15,0.024864,-0.067772,0.081673,0.039095,17.729867,53.189602,...,True,True,18,16,2026-08-21,2026-08-21,2028-12-15,2028-12-15,True,None
1,MLM-VMC,MLM,VMC,1.383114e-14,0.049987,-0.182892,0.273007,0.038179,18.155329,54.465986,...,True,True,6,4,2026-08-21,2026-08-21,2027-12-17,2027-02-19,True,None
2,HLT-AXP,HLT,AXP,-2.222718e-15,0.068528,-0.268255,0.190885,0.035402,19.579342,58.738026,...,True,True,12,17,2026-08-21,2026-08-21,2028-01-21,2028-12-15,True,None
3,SHW-HD,SHW,HD,-1.124588e-14,0.056705,-0.156361,0.178546,0.034308,20.203513,60.610538,...,True,True,7,17,2026-08-21,2026-08-21,2028-01-21,2028-12-15,True,None
4,URI-MS,URI,MS,-4.401002e-15,0.087648,-0.277321,0.274505,0.033124,20.925609,62.776828,...,True,True,12,16,2026-08-21,2026-08-21,2028-01-21,2028-12-15,True,None


In [13]:
ou_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   pair                     20 non-null     object 
 1   dependent                20 non-null     object 
 2   independent              20 non-null     object 
 3   mean                     20 non-null     float64
 4   std                      20 non-null     float64
 5   min spread               20 non-null     float64
 6   max spread               20 non-null     float64
 7   theta                    20 non-null     float64
 8   half_life                20 non-null     float64
 9   expected_convergence     20 non-null     float64
 10  hurst                    20 non-null     float64
 11  current z                20 non-null     float64
 12  stock1                   20 non-null     object 
 13  stock2                   20 non-null     object 
 14  same_issuer              20 

In [14]:
ou_results.describe()

,mean,std,min spread,max spread,theta,half_life,expected_convergence,hurst,current z,stock1_num_expirations,stock2_num_expirations
count,2.000000e+01,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,-1.859974e-15,0.061998,-0.213239,0.182916,0.030304,23.245776,69.737327,0.330218,-0.429005,10.500000,11.050000
std,7.480626e-15,0.019832,0.094164,0.061142,0.004088,2.928759,8.786276,0.025057,1.119051,5.052357,5.286278
min,-1.921576e-14,0.024864,-0.489459,0.081673,0.025187,17.729867,53.189602,0.283777,-2.280265,4.000000,4.000000
25%,-5.146663e-15,0.046829,-0.270522,0.135299,0.027510,21.319115,63.957345,0.315355,-1.150740,5.750000,5.750000
50%,-2.078812e-15,0.062988,-0.201288,0.192563,0.028903,23.983406,71.950219,0.333376,-0.156757,11.000000,13.500000
75%,1.312828e-15,0.072768,-0.152566,0.213913,0.032517,25.197311,75.591933,0.347811,0.317119,14.250000,16.000000
max,1.383114e-14,0.095808,-0.067772,0.277190,0.039095,27.519492,82.558475,0.371829,1.352716,19.000000,17.000000


In [15]:
ou_results["z_score"] = (
    ou_results["current z"]
)

ou_results[
    [
        "pair",
        "half_life",
        "theta",
        "std",
        "current z"
    ]
].head(20)

,pair,half_life,theta,std,current z
0,V-MA,17.729867,0.039095,0.024864,-0.946981
1,MLM-VMC,18.155329,0.038179,0.049987,-0.970178
2,HLT-AXP,19.579342,0.035402,0.068528,0.338066
3,SHW-HD,20.203513,0.034308,0.056705,-2.280265
4,URI-MS,20.925609,0.033124,0.087648,1.352716
5,LYV-AXP,21.450283,0.032314,0.094060,-2.174888
6,SPGI-MCO,21.528180,0.032197,0.040006,0.254589
7,MDLZ-PG,21.995870,0.031513,0.043512,1.325631
8,PEP-HSY,23.359554,0.029673,0.041325,0.129137
9,ADI-CDW,23.788271,0.029138,0.068122,0.381321


In [16]:
ENTRY_Z = 2.0

ou_results["entry_signal"] = ou_results["current z"].apply(
    lambda z: determine_entry_signal(
        z_score=z,
        entry_z=ENTRY_Z
    )
)

ou_results["signal_label"] = ou_results["entry_signal"].apply(
    signal_to_label
)

ou_results[
    [
        "pair",
        "current z",
        "entry_signal",
        "signal_label"
    ]
].head(20)

,pair,current z,entry_signal,signal_label
0,V-MA,-0.946981,0,NO_SIGNAL
1,MLM-VMC,-0.970178,0,NO_SIGNAL
2,HLT-AXP,0.338066,0,NO_SIGNAL
3,SHW-HD,-2.280265,-1,LONG_CALL
4,URI-MS,1.352716,0,NO_SIGNAL
5,LYV-AXP,-2.174888,-1,LONG_CALL
6,SPGI-MCO,0.254589,0,NO_SIGNAL
7,MDLZ-PG,1.325631,0,NO_SIGNAL
8,PEP-HSY,0.129137,0,NO_SIGNAL
9,ADI-CDW,0.381321,0,NO_SIGNAL


In [17]:
STOP_MULTIPLIER = 2.0

ou_results["stop_loss"] = ou_results.apply(
    lambda row: calculate_stop_loss(
        mean=row["mean"],
        sigma=row["std"],
        stop_multiplier=STOP_MULTIPLIER
    ),
    axis=1
)

In [18]:
ou_results["max_dte"] = ou_results["half_life"].apply(
    calculate_max_dte
)

ou_results[
    [
        "pair",
        "half_life",
        "max_dte"
    ]
].head(20)

,pair,half_life,max_dte
0,V-MA,17.729867,54
1,MLM-VMC,18.155329,55
2,HLT-AXP,19.579342,59
3,SHW-HD,20.203513,61
4,URI-MS,20.925609,63
5,LYV-AXP,21.450283,65
6,SPGI-MCO,21.528180,65
7,MDLZ-PG,21.995870,66
8,PEP-HSY,23.359554,71
9,ADI-CDW,23.788271,72


In [22]:
ANNUAL_RISK_FREE_RATE = 0.04

ou_results["discount_rate"] = (
    ANNUAL_RISK_FREE_RATE /np.sqrt(252)
)

In [23]:
ou_results["liquidation_boundary"] = ou_results.apply(
    lambda row: calculate_liquidation_boundary(
        theta=row["theta"],
        mean=row["mean"],
        sigma=row["std"],
        stop_loss=row["stop_loss"],
        discount_rate=row["discount_rate"]
    ),
    axis=1
)

RuntimeError: Could not find a liquidation boundary. Check OU parameters and discount rate.